<a href="https://colab.research.google.com/github/lsgrep/serv/blob/main/notebooks/08_rag_and_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 — RAG, and the eval harness that makes it arguable

**The claim you should be able to make when you finish:** *"When someone tells
me the model is bad, I pull forty failures and show them that thirty-four were
retrieval misses. Then we fix the cheap thing first."*

That sentence is the highest-leverage thing in this entire repo, and it needs
two artifacts behind it: a retrieval system you can measure, and an eval harness
whose numbers survive scrutiny.

### The conversation this lab is armour for

> "The answers are wrong. So the model is bad. So let's fine-tune."

Every step of that is wrong, and contradicting it directly does not work —
you would be arguing with someone's direct experience. What works is
**relocating** the problem with data:

1. Validate the observation. The answers *are* wrong.
2. Split the failures: was the right passage ever retrieved?
3. Show the split. "34 of 40 were never shown the answer."
4. Cost both paths, recommend one, name the date you review it.

Fine-tuning teaches style and format. It does not install knowledge. In those 34
cases it would have changed nothing — and being able to demonstrate that beats
asserting it.

In [ ]:
# Cell 1 — bootstrap. Runs on CPU: BM25 and the metrics need nothing exotic.
REPO, BRANCH = "https://github.com/lsgrep/serv.git", "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

from servlab import rag, evalkit as ek
from servlab.plots import use_style, SERIES, STATUS, bar_compare
use_style()
print("ready — no GPU, no API key needed for the retrieval half")

## 1. A corpus, chunked

Chunking is the first place quality is lost, and it is lost silently. Two knobs:

* **size** — too small and a passage loses the context that makes it answerable;
  too large and its embedding averages several topics into a vector that matches
  none of them well.
* **overlap** — insurance against a fact landing exactly on a boundary. Zero
  overlap is the most common reason a fact that is *in the corpus* is not
  retrievable.

Both are empirical. Measure recall, do not debate.

In [ ]:
chunks, cases = rag.policy_corpus(chunk_size=60, overlap=15)
print(f"{len(chunks)} chunks from {len(rag.POLICY_DOCS)} documents\n")
for c in chunks[:3]:
    print(f"  {c.id:<14} {c.text[:88]}...")
print(f"\n{sum(c['answerable'] for c in cases)} answerable questions, "
      f"{sum(not c['answerable'] for c in cases)} deliberately unanswerable")

Note those unanswerable questions. A golden set made only of questions the
corpus *can* answer measures half the system. Confident answers to questions the
corpus cannot support are how a RAG deployment loses trust in one screenshot,
and they are invisible unless you test for them.

## 2. Measure retrieval on its own, before anything else

`recall@k` is the ceiling on the entire system. If it is 0.6, the best
imaginable model answers 40% of questions from nothing — and no prompt
engineering, no model upgrade, and certainly no fine-tune moves that number.

Measuring retrieval separately from answer quality is the single most useful
discipline in RAG work, and most teams never do it.

In [ ]:
retriever = rag.HybridRetriever(chunks)     # BM25 only unless a dense index is added
answerable = [c for c in cases if c["answerable"]]

for k in (1, 3, 5):
    r = rag.evaluate_retrieval(retriever, answerable, k=k)
    print(f"  k={k}:  recall@{k}={r[f'recall@{k}']:.2f}  hit_rate={r['hit_rate']:.2f}  "
          f"mrr={r['mrr']:.2f}  ndcg@{k}={r[f'ndcg@{k}']:.2f}")

In [ ]:
# The aggregate hides which queries fail. Always look at the rows.
result = rag.evaluate_retrieval(retriever, answerable, k=3)
for row in result["rows"]:
    mark = "ok  " if row["hit"] else "MISS"
    print(f"  {mark}  {row['query'][:62]:<64} {row['retrieved'][:2]}")

The miss is a paraphrase: the question says *"before prices go up"*, the document
says *"notified 60 days in advance"*. No shared keywords, so BM25 cannot see it.

That is the exact failure a dense retriever fixes — and the exact reason hybrid
search exists. Lexical and dense retrieval fail on **different** queries: BM25
owns product codes, error strings and policy numbers; embeddings own paraphrase.
Fusion keeps whichever one was right without you having to know in advance.

In [ ]:
# Chunk size is empirical. Sweep it rather than picking a number from a blog post.
import matplotlib.pyplot as plt

sizes = [30, 40, 60, 80, 120, 200]
scores = []
for size in sizes:
    ch, cs = rag.policy_corpus(chunk_size=size, overlap=max(5, size // 4))
    r = rag.evaluate_retrieval(rag.HybridRetriever(ch),
                               [c for c in cs if c["answerable"]], k=3)
    scores.append(r["hit_rate"])
    print(f"  chunk size {size:>4} words -> {len(ch):>3} chunks, hit_rate {r['hit_rate']:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, scores, marker="o", color=SERIES[0])
ax.set_xlabel("chunk size (words)"); ax.set_ylabel("hit rate @3")
ax.set_title("chunk size is an empirical question, not a style choice")
ax.set_ylim(0, 1.05)
plt.show()

In [ ]:
# Retrieve wide, then rerank narrow — the highest value-per-line component in
# most RAG systems. A real reranker scores query and passage jointly; this is a
# term-coverage stand-in that shows the shape without a model download.
query = "How much notice before prices go up?"
wide = retriever.search(query, k=8)
print("BM25 top 3:")
for c, s in wide[:3]:
    print(f"   {c.id:<12} {s:5.2f}  {c.text[:60]}...")

print("\nafter reranking to 3:")
for c, s in rag.rerank_by_overlap(query, wide, top_k=3):
    print(f"   {c.id:<12} {s:5.2f}  {c.text[:60]}...")

## 3. The triage

Now the move the whole lab exists for. Given failures, split them:

* **gold passage absent from top-k** → retrieval miss. Cheap to fix: chunking,
  hybrid search, reranking. Days.
* **gold passage present, answer still wrong** → synthesis miss. Prompt, context
  ordering, or model. Weeks, and the only bucket where a model change is even
  the right *category* of fix.

The ratio *is* the recommendation.

In [ ]:
# Simulated failure set in the shape you would actually collect: 40 bad answers
# pulled from production traffic, with what was retrieved for each.
import random

rng = random.Random(0)
failures = []
for i in range(40):
    retrieval_failed = i < 34            # the split you discovered by measuring
    failures.append({
        "query": f"question {i}",
        "gold_chunk_ids": ["refunds#0"],
        "retrieved_ids": ["pricing#1", "sla#0"] if retrieval_failed else ["refunds#0", "sla#0"],
        "answer_correct": False,
    })

t = rag.triage(failures, k=3)
print(rag.triage_recommendation(t))

In [ ]:
bar_compare(["retrieval misses\n(never shown the answer)", "synthesis misses\n(shown it, still wrong)"],
            [t["retrieval_misses"], t["synthesis_misses"]],
            title="40 wrong answers, split by cause", ylabel="failures",
            highlight={"retrieval misses\n(never shown the answer)"})

### The verbatim, for the room

> "You're right that the answers are wrong — I pulled the forty worst cases. In
> thirty-four of them, the model was never shown the policy document that
> contains the answer. It's not a model-intelligence problem, it's a retrieval
> problem: the system is being asked to recall something it was never handed.
>
> That's good news. Fixing retrieval is about two weeks and cheap. Swapping or
> fine-tuning models is two months, and it wouldn't have fixed any of those
> thirty-four — fine-tuning teaches style, not facts.
>
> Here are both options costed. My recommendation is we fix retrieval, re-run
> the same two-hundred-case eval, and look at the graph together on the 30th."

Four things that sentence does: validates the observation, kills the wrong
conclusion with data rather than authority, gives exactly **one** recommendation,
and puts a date on the review. Never "the model is fine" — that argues with
their experience, and you will lose.

That last clause is a claim, and claims get probed. [Lab 10](10_finetune_what_it_teaches.ipynb) runs the experiment behind it: one LoRA trained on both a strict output format and closed-book facts, then scored separately on each. The short version to have ready — format compliance moves a lot, recall on the *same facts asked in different words* barely moves, and retrieval on the untouched base model beats it. Quote your own numbers, not these.


In [ ]:
# The probe you will get: "we already spent $80K on fine-tuning."
print("Sunk cost. The $80K is gone whether or not we use the fine-tune.\n"
      "The question is only which system is better *now*, and that is an\n"
      "afternoon's measurement, not an argument:\n")
print("  1. run the same 200-case eval against both\n"
      "  2. if the fine-tune wins, keep it — and we still fix retrieval,\n"
      "     because these 34 failures are orthogonal to which model we run\n"
      "  3. if it doesn't, we learned that for the price of an afternoon\n")
print("Note that the eval answers this without anyone having to be wrong out loud.")

## 4. The eval harness

The thing that converts *"it feels worse lately"* into a graph, and the thing
that makes a model swap an afternoon instead of a quarter.

Design, in the order you build it:

1. **100-300 golden cases** sampled from real traffic, plus deliberate edge and
   adversarial cases, plus unanswerable ones.
2. **Task-level metrics**: field accuracy, groundedness, citation validity,
   correct-refusal rate.
3. **An LLM judge for the fuzzy metrics** — *calibrated against human labels,
   with the agreement reported.* Uncalibrated, it is a vibe check with a
   spreadsheet attached.
4. **A regression gate in CI** on every prompt, model, or index change.
5. **Canary and online metrics**: escalation rate, resolution, thumbs.

In [ ]:
# Groundedness: a cheap model-free screen for invented content.
sources = [c.text for c in retriever.search("refund window", k=2)]
grounded = "Customers may request a refund within 30 days of purchase."
invented = "Refunds are issued instantly by wire transfer to any nominated account."

print(f"grounded answer : {ek.groundedness(grounded, sources):.2f}")
print(f"invented answer : {ek.groundedness(invented, sources):.2f}")
print("\nIt is a lower bound — a correct paraphrase scores low too — so use it to")
print("flag candidates for a judge or a human, never as the verdict itself.")

In [ ]:
# Citation validity. An invented citation is the worst failure there is,
# because it looks exactly like evidence.
valid_ids = [c.id for c in chunks]
print(ek.citation_validity("Per [refunds#0] the window is 30 days.", valid_ids))
print(ek.citation_validity("Per [handbook#12] the window is 90 days.", valid_ids))

In [ ]:
# Correct refusal: did it say "I don't know" on the questions the corpus
# cannot answer?
answers = ["the window is 30 days", "I don't know — that isn't covered in these documents",
           "the platform runs on AWS"]
subset = [cases[0], cases[-2], cases[-1]]
print(ek.refusal_correctness(subset, answers))
print("\nhallucinated_on_unanswerable is the number to put in front of a customer.")

## 5. Calibrate the judge before you trust a single number it produces

The step everyone skips. Label 50-100 cases by hand **once**, then measure
whether the judge agrees with you.

Raw agreement flatters a judge on skewed data — if 90% of answers are good, a
judge that says "good" every time scores 90%. Cohen's kappa subtracts that
chance agreement, and it is the number to report.

**False passes matter more than false fails.** A judge that waves bad answers
through is worse than no judge at all, because it manufactures confidence.

In [ ]:
human = [1] * 90 + [0] * 10
lazy_judge = [1] * 100                 # says "good" to everything
print("A judge that never says no:")
print(ek.calibrate_judge(human, lazy_judge))

In [ ]:
good_judge = [1] * 86 + [0] * 4 + [1] * 2 + [0] * 8
print("A judge worth running unattended:")
print(ek.calibrate_judge(human, good_judge))

## 6. How big does the eval need to be?

The humility number, and one of the fastest ways to show you understand
measurement rather than just running it.

In [ ]:
for n in (30, 60, 100, 200, 500, 1000):
    lo, hi = ek.accuracy_ci(0.80, n)
    print(f"  n={n:>5}:  80% accuracy, 95% CI [{lo:.1%}, {hi:.1%}]  "
          f"(+/- {(hi-lo)/2*100:.1f} points)")

print()
for delta in (0.10, 0.05, 0.03, 0.01):
    print(f"  to detect a {delta:.0%} change from 80%: "
          f"~{ek.min_samples_for_detectable_difference(0.80, delta):,} cases")

Read that second block again. **Detecting a 3-point difference needs thousands
of cases.** So when someone reports that a change improved accuracy from 81% to
84% on a 200-case suite, the correct response is not celebration:

> "That's inside the noise band for a 200-case eval — the confidence interval is
> about ±5.5 points either side. It's not evidence the change hurt, but it isn't
> evidence it helped either. If this decision matters, we need either more cases
> or a paired comparison on the same items, which is far more sensitive."

Paired comparison is the practical escape: comparing two systems **on the same
cases** removes the case-difficulty variance and detects much smaller differences
than the independent-sample maths above suggests.

In [ ]:
# Bootstrap error bars on any per-case metric, not just proportions.
scores = [1.0] * 82 + [0.0] * 18
lo, hi = ek.bootstrap_ci(scores, n_resamples=2000)
print(f"82% on 100 cases -> bootstrap 95% CI [{lo:.1%}, {hi:.1%}]")

## 7. The regression gate

The artifact that turns the eval from a report into a control. Set tolerances
**above the noise floor** — otherwise the gate fires on sampling noise and gets
switched off within a month, which is worse than never having built it.

In [ ]:
baseline = {"accuracy": 0.86, "groundedness": 0.91, "correct_refusal_rate": 0.95}

candidate_bad = {"accuracy": 0.71, "groundedness": 0.90, "correct_refusal_rate": 0.94}
print(ek.regression_gate(baseline, candidate_bad, {"accuracy": 0.03}, n=200))
print()

candidate_noise = {"accuracy": 0.83, "groundedness": 0.92, "correct_refusal_rate": 0.95}
print(ek.regression_gate(baseline, candidate_noise, {"accuracy": 0.02}, n=200))

Notice the second one: it fails, **and it tells you the failure is inside two
standard errors.** A gate that explains itself is a gate people keep.

In CI this runs on every prompt change, model change, and index rebuild. The
judge runs on a cheap-tier model, so a full 200-case pass costs pennies.

### The funding pitch, verbatim

> "Right now, quality is decided by whoever ran the most recent demo. For two
> engineer-weeks we get a two-hundred-question exam the system has to pass before
> any change ships. It turns 'it feels worse lately' into a graph. It's what lets
> us upgrade models the day a better one launches instead of six months later.
> And candidly, it's the difference between an AI feature and an AI liability."

The adoption trick: if the execs have been trying prompts on Friday afternoons,
**make their Friday prompts the seed of the golden set.** Their judgment becomes
the benchmark, so the process is institutionalised rather than replaced — and
nobody has to be told their method was wrong.

## What to be able to say afterwards

1. **Fine-tuning does not add knowledge.** Retrieval does. Diagnose before
   prescribing, and the diagnosis is a two-bucket split you can run in an hour.
2. **Measure retrieval separately** — recall@k is the ceiling on everything
   downstream, and most teams never look at it.
3. **Hybrid search works because lexical and dense fail on different queries**,
   and reranking is the cheapest large win available.
4. **An uncalibrated LLM judge is a vibe check with a spreadsheet.** Report
   agreement and kappa, and weight false passes more heavily.
5. **Say the confidence interval out loud** before anyone celebrates three points.
6. **The eval suite is the exit option** — it is what makes swapping a model an
   afternoon instead of a quarter, which is the honest answer to lock-in.

**Next:** [lab 9](09_serving_levers.ipynb) measures the serving optimisations
that lab 1's diagnosis playbook only names.